# 🎬 Anime Video Renderizer (Kaggle)

Este notebook foi feito para pegar os arquivos exportados pelo **VideoRender Editor** e processá-los na nuvem usando FFmpeg.

### Passos:
1. Crie um Dataset no Kaggle (ex: nomeado `arquivos`) e faça upload dos seus arquivos:
   - **Vídeo Original** (`.mp4`)
   - **Áudio Dublado** (`.mp3` ou `.wav`)
   - **Legenda** (`legendas.ass`)
   - **Configuração** (`videorender-project.json`)
2. Configure os nomes corretos na célula abaixo.
3. Rode todas as células!

In [1]:
# Instalar/Atualizar o FFmpeg no ambiente do Kaggle (Ubuntu)
!apt-get update && apt-get install -y ffmpeg
!ffmpeg -version

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]       
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,546 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [1

In [2]:
import json
import subprocess
import os
import base64
import shutil

# Nomes exatos que aparecem na barra lateral do seu print
VIDEO_FILENAME  = "SnapVid.Net_7627800078765362486-hd.mp4" 
AUDIO_FILENAME  = "VIDEO_COMPLETO_FINAL (3).mp3"            
ASS_FILENAME    = "legendas.ass"
CONFIG_FILENAME = "videorender-project.json"

WORKING_DIR = "/kaggle/working/"

# Função para encontrar o caminho absoluto do arquivo em qualquer subpasta do Kaggle
def find_file(filename):
    for root, dirs, files in os.walk('/kaggle/input/'):
        if filename in files:
            return os.path.join(root, filename)
    return None

VIDEO_INPUT = find_file(VIDEO_FILENAME)
AUDIO_INPUT = find_file(AUDIO_FILENAME)
CONFIG_FILE = find_file(CONFIG_FILENAME)
ASS_ORIGINAL = find_file(ASS_FILENAME)
OUTPUT_FILE = os.path.join(WORKING_DIR, "video_final_renderizado.mp4")

# Verificações
if not VIDEO_INPUT: print(f"❌ Vídeo não encontrado: {VIDEO_FILENAME}")
if not AUDIO_INPUT: print(f"❌ Áudio não encontrado: {AUDIO_FILENAME}")
if not CONFIG_FILE: print(f"❌ Configuração não encontrada: {CONFIG_FILENAME}")

if ASS_ORIGINAL:
    ASS_LOCAL = os.path.join(WORKING_DIR, "legendas_temp.ass")
    shutil.copy2(ASS_ORIGINAL, ASS_LOCAL)
    print("✅ Legenda encontrada e copiada para ambiente de trabalho.")
else:
    print(f"❌ ERRO: Legenda não encontrada: {ASS_FILENAME}")

if CONFIG_FILE:
    with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
        config = json.load(f)
    print("✅ Configurações do projeto carregadas com sucesso!")


✅ Legenda encontrada e copiada para ambiente de trabalho.
✅ Configurações do projeto carregadas com sucesso!


In [3]:
def build_ffmpeg_command(config, video_in, audio_in, ass_in, out_file):
    filters = []
    last_stream = "[0:v]"
    overlay_inputs = []
    
    # 1. Resolução e Formato
    out_format = config['video'].get('outputFormat', '16:9')
    out_w, out_h = (1080, 1920) if out_format == '9:16' else (1920, 1080)
    
    # 2. Crop / Zoom
    crop_info = config.get('cropZoom', {})
    if crop_info.get('enabled'):
        zs = crop_info.get('zoomStart', 1.0)
        fx = crop_info.get('focusX', 0.5)
        fy = crop_info.get('focusY', 0.5)
        
        in_w = config['video']['info']['width']
        in_h = config['video']['info']['height']
        
        cw = int(in_w / zs)
        ch = int(in_h / zs)
        cx = int((in_w - cw) * fx)
        cy = int((in_h - ch) * fy)
        
        filters.append(f"{last_stream}crop={cw}:{ch}:{cx}:{cy},scale={out_w}:{out_h}[cropped]")
        last_stream = "[cropped]"
    else:
        filters.append(f"{last_stream}scale={out_w}:{out_h}:force_original_aspect_ratio=increase,crop={out_w}:{out_h}[scaled]")
        last_stream = "[scaled]"
        
    # 3. Tratamento de Cor (Color Grading)
    cg = config.get('colorGrade', {})
    if cg:
        b = cg.get('brightness', 0) / 100.0
        c = 1.0 + cg.get('contrast', 0) / 100.0
        s = 1.0 + cg.get('saturation', 0) / 100.0
        g = cg.get('gamma', 1.0)
        filters.append(f"{last_stream}eq=brightness={b}:contrast={c}:saturation={s}:gamma={g}[colorgraded]")
        last_stream = "[colorgraded]"

        temp = cg.get('temperature', 0)
        if temp != 0:
            red_mod = temp / 100.0 if temp > 0 else 0
            blue_mod = abs(temp) / 100.0 if temp < 0 else 0
            filters.append(f"{last_stream}colorbalance=rm={red_mod}:bm={blue_mod}[temp_applied]")
            last_stream = "[temp_applied]"

        sharp = cg.get('sharpness', 1.0)
        if sharp > 1.0:
            amount = sharp - 1.0
            filters.append(f"{last_stream}unsharp=5:5:{amount}:5:5:0.0[sharpened]")
            last_stream = "[sharpened]"
        
        v = cg.get('vignette', 0)
        if v > 0:
            filters.append(f"{last_stream}vignette=a={v}[vignetted]")
            last_stream = "[vignetted]"

    # 4. Overlays (Marcas d'água, Imagens)
    overlays = config.get('overlays', [])
    for i, ov in enumerate(overlays):
        if ov['type'] in ['image', 'watermark']:
            header, encoded = ov['content'].split(",", 1)
            ext = header.split(";")[0].split("/")[1]
            filename = f"/kaggle/working/temp_overlay_{i}.{ext}"
            
            with open(filename, "wb") as f:
                f.write(base64.b64decode(encoded))
                
            overlay_inputs.append(filename)
            input_idx = len(overlay_inputs) + 1
            
            ox = int((ov['x'] / 100.0) * out_w)
            oy = int((ov['y'] / 100.0) * out_h)
            ow = int((ov['width'] / 100.0) * out_w)
            oh = int((ov['height'] / 100.0) * out_h)
            opacity = ov.get('opacity', 1.0)
            
            filters.append(f"[{input_idx}:v]scale={ow}:{oh}[ov_scaled_{i}]")
            alpha_filter = f",colorchannelmixer=aa={opacity}" if opacity < 1.0 else ""
            filters.append(f"[ov_scaled_{i}]format=rgba{alpha_filter}[ov_alpha_{i}]")
            
            time_filter = ""
            tin = ov.get('timeIn', 0)
            tout = ov.get('timeOut', 0)
            if tin > 0 or tout > 0:
                if tout == 0: tout = 999999
                time_filter = f":enable='between(t,{tin},{tout})'"
                
            filters.append(f"{last_stream}[ov_alpha_{i}]overlay=x={ox}:y={oy}{time_filter}[with_ov_{i}]")
            last_stream = f"[with_ov_{i}]"
            
        elif ov['type'] == 'text':
            ox = int((ov['x'] / 100.0) * out_w)
            oy = int((ov['y'] / 100.0) * out_h)
            txt = ov['content'].replace("'", "\\'").replace(":", "\\:")
            fsize = int((ov.get('fontSize', 32) * (out_h / 1080)))
            fcolor = ov.get('fontColor', '#ffffff')
            
            time_filter = ""
            tin = ov.get('timeIn', 0)
            tout = ov.get('timeOut', 0)
            if tin > 0 or tout > 0:
                if tout == 0: tout = 999999
                time_filter = f":enable='between(t,{tin},{tout})'"
                
            filters.append(f"{last_stream}drawtext=text='{txt}':x={ox}:y={oy}:fontsize={fsize}:fontcolor={fcolor}{time_filter}[with_txt_{i}]")
            last_stream = f"[with_txt_{i}]"
            
    # 5. Legendas Animadas .ASS
    filters.append(f"{last_stream}ass='legendas_temp.ass'[subbed]")
    last_stream = "[subbed]"
    
    filter_complex = ";".join(filters)
    
    cmd = [
        "ffmpeg", "-y",
        "-threads", "4",          # Força o uso de múltiplas threads do Kaggle
        "-filter_threads", "4",   # Acelera os filtros de imagem e legenda
        "-i", video_in,
        "-i", audio_in,
    ]
    
    for ov_file in overlay_inputs:
        cmd.extend(["-i", ov_file])
        
    cmd.extend([
        "-filter_complex", filter_complex,
        "-map", last_stream,    
        "-map", "1:a",          
        "-c:v", "h264_nvenc",     # Força o uso do chip da placa de vídeo (NVENC)
        "-cq", "18",              # É o equivalente ao "crf" para a placa de vídeo
        "-preset", "p6",          # Preset oficial da NVIDIA (P6 é focado em qualidade e velocidade)
        "-c:a", "aac",          
        "-b:a", "192k",         
        "-shortest",            
        out_file
    ])

    
    return cmd

os.chdir(WORKING_DIR)
command = build_ffmpeg_command(config, VIDEO_INPUT, AUDIO_INPUT, ASS_LOCAL, OUTPUT_FILE)
print("🛠️ Comando FFmpeg gerado:")
print(" ".join(command))


🛠️ Comando FFmpeg gerado:
ffmpeg -y -threads 4 -filter_threads 4 -i /kaggle/input/datasets/alessandroalvs/arquivos/SnapVid.Net_7627800078765362486-hd.mp4 -i /kaggle/input/datasets/alessandroalvs/arquivos/VIDEO_COMPLETO_FINAL (3).mp3 -filter_complex [0:v]crop=1745:981:87:0,scale=1920:1080[cropped];[cropped]eq=brightness=0.05:contrast=1.33:saturation=1.34:gamma=1.05[colorgraded];[colorgraded]colorbalance=rm=0:bm=0.05[temp_applied];[temp_applied]unsharp=5:5:1.0:5:5:0.0[sharpened];[sharpened]vignette=a=0.15[vignetted];[vignetted]ass='legendas_temp.ass'[subbed] -map [subbed] -map 1:a -c:v h264_nvenc -cq 18 -preset p4 -c:a aac -b:a 192k -shortest /kaggle/working/video_final_renderizado.mp4


In [4]:
# Execute esta célula para INICIAR A RENDERIZAÇÃO NO KAGGLE!
print("🚀 Iniciando renderização (Isso pode demorar dependendo do tamanho do vídeo)...")

process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)

# Exibir progresso ao vivo
for line in process.stderr:
    if "frame=" in line:
        print(line.strip(), end='\r')

process.wait()

if process.returncode == 0:
    print("\n\n🎉 Renderização concluída com sucesso!")
    print(f"📁 O arquivo foi salvo na pasta Output (working) do Kaggle com o nome: {OUTPUT_FILE.split('/')[-1]}")
else:
    print("\n\n❌ Ocorreu um erro durante a renderização:")
    print(process.stderr.read())


🚀 Iniciando renderização (Isso pode demorar dependendo do tamanho do vídeo)...


KeyboardInterrupt: 

In [5]:
from IPython.display import FileLink
import os

# Nome do arquivo (o FileLink busca direto na pasta /kaggle/working/ por padrão)
arquivo_saida = "video_final_renderizado.mp4"

if os.path.exists(arquivo_saida):
    print("✅ Arquivo encontrado! Clique no link abaixo para baixar:")
    display(FileLink(arquivo_saida))
else:
    print(f"❌ Arquivo não encontrado. Verifique se a renderização terminou corretamente e gerou o '{arquivo_saida}'.")

✅ Arquivo encontrado! Clique no link abaixo para baixar:


/kaggle/working/video_final_renderizado.mp4